# 2. Assemble trip-level Citi Bike data (2013–2019)

**Who worked on this:** Rune Lemmens

This notebook independently assembles the downloaded yearly archives into the unified plain CSV. Run 01_download.ipynb first so the required ZIP archives exist in `data/raw`.

In [5]:
import json
import os
import re
import shutil
import tempfile
import time
from collections import defaultdict
from contextlib import contextmanager
from pathlib import Path
from zipfile import BadZipFile, ZipFile

import pandas as pd

# Standalone paths and the shared 2013–2019 schema required by Section 3.
START_YEAR, END_YEAR = 2013, 2019
DATA_DIR = Path.cwd() / "data"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_PATH = DATA_DIR / "processed" / "citibike_trips_unified_2013_2019.csv"
ASSEMBLY_STATE_PATH = OUTPUT_PATH.with_suffix(".manifest.json")
MEMBER_TEMP_PATH = OUTPUT_PATH.with_suffix(".member.tmp")
CHUNK_SIZE = 200_000
PROGRESS_INTERVAL_SECONDS = 3
MONTH_KEY_RE = re.compile(r"(\d{6})-citibike-tripdata")
DATE_CODE_RE = re.compile(r"^(\d{4})(\d{2})?-citibike-tripdata(?:\.csv)?\.zip$")
UNIFIED_COLUMNS = [
    "started_at", "ended_at", "start_station_id", "start_station_name", "start_lat", "start_lng",
    "end_station_id", "end_station_name", "end_lat", "end_lng", "member_casual", "birth_year", "gender", "trip_duration_s",
]
OLD_SCHEMA_RENAME = {
    "starttime": "started_at", "start_time": "started_at", "stoptime": "ended_at", "stop_time": "ended_at",
    "start_station_id": "start_station_id", "start_station_name": "start_station_name",
    "start_station_latitude": "start_lat", "start_station_longitude": "start_lng",
    "end_station_id": "end_station_id", "end_station_name": "end_station_name",
    "end_station_latitude": "end_lat", "end_station_longitude": "end_lng", "birth_year": "birth_year", "gender": "gender",
}
OLD_SCHEMA_MARKERS = ("starttime", "start_time")
USERTYPE_TO_MEMBER_CASUAL = {"subscriber": "member", "customer": "casual"}

# Archive readers are repeated here so this notebook does not rely on a prior kernel.
def _is_csv_member(name: str) -> bool:
    base = Path(name).name
    return base.lower().endswith(".csv") and "__MACOSX" not in name and not base.startswith("._") and base != ".DS_Store"


def _nested_zip_csv_info(source_zip: ZipFile, nested_zip_name: str) -> list[tuple[str, int, int]]:
    with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as temp_file:
        temp_path = Path(temp_file.name)
        with source_zip.open(nested_zip_name) as nested_stream:
            shutil.copyfileobj(nested_stream, temp_file)
    try:
        with ZipFile(temp_path) as nested_zip:
            return [(info.filename, info.file_size, info.CRC) for info in nested_zip.infolist() if _is_csv_member(info.filename)]
    finally:
        temp_path.unlink(missing_ok=True)


def discover_source_csvs(source_zip: ZipFile) -> list[tuple[str, str | None]]:
    grouped_sources = defaultdict(list)
    for info in source_zip.infolist():
        name = info.filename
        if _is_csv_member(name):
            base = Path(name).name
            match = MONTH_KEY_RE.search(base)
            month_key = match.group(1) if match else base
            grouped_sources[month_key].append(((name, None), base, True))
        elif Path(name).name.lower().endswith(".zip") and "__MACOSX" not in name and not Path(name).name.startswith("._"):
            match = MONTH_KEY_RE.search(Path(name).name)
            nested_month = match.group(1) if match else None
            for inner_name, _, _ in _nested_zip_csv_info(source_zip, name):
                inner_match = MONTH_KEY_RE.search(Path(inner_name).name)
                month_key = inner_match.group(1) if inner_match else nested_month or Path(inner_name).name
                grouped_sources[month_key].append(((name, inner_name), Path(inner_name).name, False))
    chosen = []
    for month_key, candidates in grouped_sources.items():
        flat = [item for item in candidates if item[2] and item[1] == f"{month_key}-citibike-tripdata.csv"]
        direct = [item for item in candidates if item[2]]
        selected = flat if flat else direct if direct else candidates
        chosen.extend(descriptor for descriptor, _, _ in selected)
    return chosen


@contextmanager
def open_source_csv(archive_path: Path, descriptor: tuple[str, str | None]):
    outer_member, inner_member = descriptor
    with ZipFile(archive_path) as source_zip:
        if inner_member is None:
            with source_zip.open(outer_member) as csv_stream:
                yield csv_stream
            return
        with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as temp_file:
            temp_path = Path(temp_file.name)
            with source_zip.open(outer_member) as nested_stream:
                shutil.copyfileobj(nested_stream, temp_file)
        try:
            with ZipFile(temp_path) as nested_zip, nested_zip.open(inner_member) as csv_stream:
                yield csv_stream
        finally:
            temp_path.unlink(missing_ok=True)

In [6]:
# Independently recover the in-scope archive inventory from S3 for size validation.
import xml.etree.ElementTree as ET
import requests

BUCKET_URL = "https://s3.amazonaws.com/tripdata/"
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

def list_main_system_files() -> list[tuple[str, int]]:
    files = []
    continuation_token = None
    while True:
        params = {"list-type": "2"}
        if continuation_token:
            params["continuation-token"] = continuation_token
        response = requests.get(BUCKET_URL, params=params, timeout=30)
        response.raise_for_status()
        root = ET.fromstring(response.content)
        for contents in root.findall("s3:Contents", NS):
            key = contents.findtext("s3:Key", namespaces=NS)
            size = contents.findtext("s3:Size", namespaces=NS)
            if not key or not size or key.startswith("JC-"):
                continue
            match = DATE_CODE_RE.match(key)
            if match and START_YEAR <= int(match.group(1)) <= END_YEAR:
                files.append((key, int(size)))
        continuation_token = root.findtext("s3:NextContinuationToken", namespaces=NS)
        if not continuation_token:
            break
    return sorted(files)

files_to_fetch = list_main_system_files()
print(f"Found {len(files_to_fetch)} in-scope archives in S3 for validation.")

Found 7 in-scope archives in S3 for validation.


## 3. Assemble trip-level data (2013–2019)

Combines all in-scope source trips from 2013 through 2019 into a standardized plain CSV. The two legacy schemas are normalized to the same columns; 2020+ archives are excluded. Assembly can resume from completed source CSVs.

In [7]:
def normalize_columns(columns: list[str]) -> list[str]:
    """Standardize source headers before applying legacy schema mappings."""
    return [str(column).strip().lower().replace(" ", "_") for column in columns]


def select_members_for_zip(source_zip: ZipFile) -> list[tuple[str, str | None]]:
    """Choose unique monthly CSV sources, preferring flat copies over nested copies."""
    return discover_source_csvs(source_zip)


def source_member_metadata(
    source_zip: ZipFile, descriptor: tuple[str, str | None]
) -> tuple[int, int]:
    """Return uncompressed byte size and CRC for a direct or nested source CSV."""
    outer_member, inner_member = descriptor
    if inner_member is None:
        info = source_zip.getinfo(outer_member)
        return info.file_size, info.CRC

    for member_name, file_size, crc in _nested_zip_csv_info(source_zip, outer_member):
        if member_name == inner_member:
            return file_size, crc
    raise KeyError(f"Nested CSV {inner_member!r} was not found in {outer_member!r}")


def map_chunk_to_unified(chunk: pd.DataFrame) -> pd.DataFrame:
    """Normalize one of the two legacy source schemas into the 2013–2019 output."""
    chunk = chunk.copy()
    chunk.columns = normalize_columns(list(chunk.columns))
    if not any(marker in chunk.columns for marker in OLD_SCHEMA_MARKERS):
        raise ValueError(f"Expected a 2013–2019 legacy schema, got: {list(chunk.columns)}")

    source_duration = next(
        (
            pd.to_numeric(chunk[column], errors="coerce")
            for column in ("tripduration", "trip_duration")
            if column in chunk.columns
        ),
        None,
    )
    mapped = chunk.rename(columns=OLD_SCHEMA_RENAME)
    usertype_col = "usertype" if "usertype" in mapped.columns else "user_type"
    mapped["member_casual"] = (
        mapped.get(usertype_col, pd.Series(index=mapped.index, dtype="object"))
        .astype("string").str.strip().str.lower().map(USERTYPE_TO_MEMBER_CASUAL)
    )

    for column in UNIFIED_COLUMNS:
        if column not in mapped.columns:
            mapped[column] = pd.NA
    for column in ("start_station_id", "end_station_id"):
        mapped[column] = mapped[column].astype("string").str.replace(r"\.0$", "", regex=True)
    for column in ("start_lat", "start_lng", "end_lat", "end_lng", "birth_year", "gender"):
        mapped[column] = pd.to_numeric(mapped[column], errors="coerce")

    mapped["started_at"] = pd.to_datetime(mapped["started_at"], format="mixed", errors="coerce")
    mapped["ended_at"] = pd.to_datetime(mapped["ended_at"], format="mixed", errors="coerce")
    timestamp_duration = (mapped["ended_at"] - mapped["started_at"]).dt.total_seconds()
    mapped["trip_duration_s"] = (
        source_duration if source_duration is not None else timestamp_duration
    )
    return mapped[UNIFIED_COLUMNS]


def index_zip_members(
    zip_paths: list[Path],
) -> dict[Path, tuple[list[tuple[str, str | None]], dict[tuple[str, str | None], int], dict[tuple[str, str | None], int], int]]:
    """Pre-index selected sources, uncompressed sizes, and CRCs."""
    index = {}
    for zip_path in zip_paths:
        try:
            with ZipFile(zip_path) as zf:
                members = select_members_for_zip(zf)
                metadata = {member: source_member_metadata(zf, member) for member in members}
                sizes = {member: metadata[member][0] for member in members}
                crcs = {member: metadata[member][1] for member in members}
            index[zip_path] = (members, sizes, crcs, sum(sizes.values()))
        except BadZipFile as exc:
            raise ValueError(f"{zip_path.name} is not a valid ZIP; rerun the download cell for that archive.") from exc
    return index

In [8]:
# Assemble only fully downloaded archives in the configured 2013–2019 window.
expected_sizes = dict(files_to_fetch)
local_zip_paths = {path.name: path for path in RAW_DIR.glob("*.zip") if DATE_CODE_RE.match(path.name)}
missing_files = set(expected_sizes) - set(local_zip_paths)
wrong_size_files = [
    name for name, expected_size in expected_sizes.items()
    if name in local_zip_paths and local_zip_paths[name].stat().st_size != expected_size
]
if missing_files or wrong_size_files:
    raise FileNotFoundError(
        "Cannot assemble an incomplete download. "
        f"Missing: {sorted(missing_files)[:5]}; wrong size: {wrong_size_files[:5]}. "
        "Run the download cell to finish or repair these archives."
    )

zip_paths = sorted(local_zip_paths[name] for name in expected_sizes)
zip_index = index_zip_members(zip_paths)
source_records = []
for zip_path in zip_paths:
    members, sizes, crcs, _ = zip_index[zip_path]
    for descriptor in members:
        outer_member, inner_member = descriptor
        member_name = inner_member or outer_member
        member_match = MONTH_KEY_RE.search(Path(member_name).name)
        if member_match:
            date_order = int(member_match.group(1))
        else:
            archive_match = DATE_CODE_RE.match(zip_path.name)
            date_order = int(archive_match.group(1) + (archive_match.group(2) or "00"))
        source_id = json.dumps([zip_path.name, outer_member, inner_member], separators=(",", ":"))
        source_records.append((date_order, zip_path.name, descriptor, sizes[descriptor], crcs[descriptor], source_id))
source_records.sort(key=lambda item: (item[0], item[1], item[2][0], item[2][1] or ""))
current_source_fingerprints = {
    source_id: {"size": source_size, "crc32": crc}
    for _, _, _, source_size, crc, source_id in source_records
}
current_source_order = [record[5] for record in source_records]


def save_assembly_state(state: dict) -> None:
    state_temp = ASSEMBLY_STATE_PATH.with_suffix(ASSEMBLY_STATE_PATH.suffix + ".tmp")
    state_temp.write_text(json.dumps(state, indent=2), encoding="utf-8")
    state_temp.replace(ASSEMBLY_STATE_PATH)


def reset_assembly_state() -> dict:
    OUTPUT_PATH.unlink(missing_ok=True)
    ASSEMBLY_STATE_PATH.unlink(missing_ok=True)
    MEMBER_TEMP_PATH.unlink(missing_ok=True)
    return {
        "format_version": 1,
        "source_fingerprints": {},
        "completed_sources": [],
        "rows_by_source": {},
        "committed_size": 0,
        "header_written": False,
    }


try:
    assembly_state = json.loads(ASSEMBLY_STATE_PATH.read_text(encoding="utf-8"))
except (FileNotFoundError, json.JSONDecodeError, OSError):
    assembly_state = None

if not assembly_state or assembly_state.get("format_version") != 1:
    assembly_state = reset_assembly_state()

completed_sources = assembly_state.get("completed_sources", [])
completed_set = set(completed_sources)
source_fingerprints = assembly_state.get("source_fingerprints", {})
old_members_changed = any(
    source_id not in current_source_fingerprints
    or current_source_fingerprints[source_id] != source_fingerprints.get(source_id)
    for source_id in completed_sources
)
completed_in_order = [source_id for source_id in current_source_order if source_id in completed_set]
new_sources_before_checkpoint = completed_in_order != current_source_order[:len(completed_in_order)]
if old_members_changed or new_sources_before_checkpoint:
    print("[reset] Source set changed; rebuilding the scoped 2013–2019 output chronologically.")
    assembly_state = reset_assembly_state()
    completed_set = set()

assembly_state["source_fingerprints"].update(current_source_fingerprints)
committed_size = int(assembly_state.get("committed_size", 0))
if not OUTPUT_PATH.exists() and committed_size:
    raise RuntimeError("Assembly checkpoint exists but output is missing; delete checkpoint and rebuild.")
if OUTPUT_PATH.exists():
    actual_size = OUTPUT_PATH.stat().st_size
    if actual_size < committed_size:
        raise RuntimeError("Output is shorter than its checkpoint; delete output and manifest to rebuild.")
    if actual_size > committed_size:
        with open(OUTPUT_PATH, "r+b") as output_file:
            output_file.truncate(committed_size)
save_assembly_state(assembly_state)

bytes_total = sum(record[3] for record in source_records)
bytes_done = 0
total_rows_this_run = 0
last_print = time.monotonic()

for date_order, archive_name, descriptor, uncompressed_size, crc, source_id in source_records:
    if source_id in completed_set:
        bytes_done += uncompressed_size
        continue

    MEMBER_TEMP_PATH.unlink(missing_ok=True)
    row_count_for_source = 0
    header_written = bool(assembly_state["header_written"])
    with open(MEMBER_TEMP_PATH, "w", encoding="utf-8", newline="") as staged_output:
        with open_source_csv(local_zip_paths[archive_name], descriptor) as source:
            for chunk in pd.read_csv(source, chunksize=CHUNK_SIZE, low_memory=False):
                unified = map_chunk_to_unified(chunk)
                unified.to_csv(staged_output, index=False, header=not header_written)
                header_written = True
                row_count_for_source += len(unified)
                total_rows_this_run += len(unified)
                now = time.monotonic()
                if now - last_print >= PROGRESS_INTERVAL_SECONDS:
                    pct = (bytes_done + source.tell()) / bytes_total * 100 if bytes_total else 0
                    print(f"    ... {pct:5.1f}% assembled, {source_id}")
                    last_print = now

    if row_count_for_source:
        with open(OUTPUT_PATH, "ab") as final_output, open(MEMBER_TEMP_PATH, "rb") as staged_input:
            shutil.copyfileobj(staged_input, final_output)
            final_output.flush()
            os.fsync(final_output.fileno())
        assembly_state["header_written"] = True

    assembly_state["completed_sources"].append(source_id)
    assembly_state["rows_by_source"][source_id] = row_count_for_source
    assembly_state["committed_size"] = OUTPUT_PATH.stat().st_size if OUTPUT_PATH.exists() else 0
    save_assembly_state(assembly_state)
    completed_set.add(source_id)
    bytes_done += uncompressed_size
    MEMBER_TEMP_PATH.unlink(missing_ok=True)
    print(f"[saved] {source_id} ({row_count_for_source:,} trips; will skip on rerun)")

if not assembly_state["header_written"]:
    raise RuntimeError("No trips were assembled; check the in-scope source archives.")

OUTPUT_PATH.touch(exist_ok=True)
total_rows = sum(assembly_state["rows_by_source"].values())
output_size_gb = OUTPUT_PATH.stat().st_size / 1_000_000_000
print(f"\n2013–2019 assembly complete: {total_rows:,} trips across {len(completed_set):,} CSV sources.")
print(f"Output: {OUTPUT_PATH} ({output_size_gb:.2f} GB).")


2013–2019 assembly complete: 91,944,421 trips across 113 CSV sources.
Output: c:\Users\lemme\OneDrive\Desktop\2025_School\deep learning\Ctrl-Alt-Defeat\NYC Citi Bike System Data\data\processed\citibike_trips_unified_2013_2019.csv (15.03 GB).
